In [1]:
from sentence_transformers import SentenceTransformer 
import faiss
import numpy as np

/opt/anaconda3/envs/ml-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#just to simplify the product names a bit we can do this 

import pandas as pd
import re

# load your csv
df = pd.read_csv("products.csv")


# remove only packaging + marketing fluff words
REMOVE_WORDS = {
    # packaging / format
    "oz", "lb", "lbs", "fl", "ct", "pk", "pack", "count",
    "size", "mini", "large", "small", "medium", "giant",
    "big", "value", "family", "party", "individual",
    "single", "bulk", "bag", "box", "bottle", "can",
    "jar", "tub", "pouch", "packet", "carton",
    "container", "case", "set", "kit", "bundle",

    # marketing fluff
    "premium", "classic", "original", "traditional",
    "authentic", "natural", "all", "real", "pure",
    "simply", "just", "good", "best", "great",
    "delicious", "tasty", "fresh", "homestyle",
    "wholesome", "everyday", "favorite", "select",
    "choice", "special", "ultimate", "finest"
}


def clean_product_name(name):

    # lowercase
    name = str(name).lower()

    # remove special characters
    name = re.sub(r'[^a-zA-Z0-9\s]', ' ', name)

    # split into words
    words = name.split()

    # remove noise words
    cleaned_words = [word for word in words if word not in REMOVE_WORDS]

    # join back
    cleaned_name = " ".join(cleaned_words)

    # remove extra whitespace
    cleaned_name = re.sub(r'\s+', ' ', cleaned_name).strip()

    return cleaned_name


# assuming your product name column is called "product_name"
df["cleaned_name"] = df["product_name"].apply(clean_product_name)


# preview
print(df[["product_name", "cleaned_name"]].head())  #keeping both og name & clean name as og will be used for displaying and clean for rag


# save cleaned csv
df.to_csv("cleaned_products.csv", index=False)

                                        product_name  \
0                         Chocolate Sandwich Cookies   
1                                   All-Seasons Salt   
2               Robust Golden Unsweetened Oolong Tea   
3  Smart Ones Classic Favorites Mini Rigatoni Wit...   
4                          Green Chile Anytime Sauce   

                                        cleaned_name  
0                         chocolate sandwich cookies  
1                                       seasons salt  
2               robust golden unsweetened oolong tea  
3  smart ones favorites rigatoni with vodka cream...  
4                          green chile anytime sauce  


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

# load embedding model (like in cnn we used mobilenetv2 were using this)
model = SentenceTransformer('all-MiniLM-L6-v2')


# get cleaned product names as list
product_names = df["cleaned_name"].tolist()


# generate embeddings
embeddings = model.encode(  #here for generating embeddings its .encode and in cnn it was .predict
    product_names,
    show_progress_bar=True
)


# convert to numpy float32
embeddings = np.array(embeddings).astype("float32")


# check shape
print(embeddings.shape) #minilm converts each word into vectors of len 384 

Batches: 100%|██████████| 1553/1553 [00:29<00:00, 52.53it/s]

(49688, 384)


In [4]:
# L2 normalization (imp for cosine similarity otw itll give similarity based on nos and not direction )
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1  #if embeddings contains all 0 then norm will be 0 so to prevent divison by 0 below we make it 1
embeddings = embeddings / norms

print("Normalized embeddings ready")

Normalized embeddings ready


In [5]:
# embedding dimension
d = embeddings.shape[1]

index = faiss.IndexFlatIP(d)

# FAISS expects float32
embeddings = embeddings.astype('float32')

# add embeddings to FAISS index
index.add(embeddings)

print("FAISS index built")

FAISS index built


In [6]:
'''
#now well save embeddings, faiss index and prod names 

# save embeddings
np.save("product_embeddings.npy", embeddings)

# save FAISS index
faiss.write_index(index, "products.index")

# save cleaned dataframe
df.to_csv("cleaned_products.csv", index=False)

print("Everything saved successfully")

'''

Everything saved successfully


In [ ]:
#if u want to load later on do this:

embeddings = np.load("product_embeddings.npy")

index = faiss.read_index("products.index")

df = pd.read_csv("cleaned_products.csv")


In [7]:
def retrieve_product(query, k=5):

    # encode query
    query_embedding = model.encode([query])

    # convert to float32
    query_embedding = np.array(query_embedding).astype("float32")

    # normalize query embedding
    faiss.normalize_L2(query_embedding)

    # search FAISS index
    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        product = df.iloc[idx]

        results.append({
            "product_name": product["product_name"],
            "cleaned_name": product["cleaned_name"],
            "score": float(score)
        })

    return results

In [10]:
results = retrieve_product("pizza cheese")

for r in results:
    print(r)

{'product_name': 'Pizza, Classic Cheese', 'cleaned_name': 'pizza cheese', 'score': 1.0}
{'product_name': 'Cheese Party Pizza', 'cleaned_name': 'cheese pizza', 'score': 0.9758967161178589}
{'product_name': 'Original Cheese Pizza', 'cleaned_name': 'cheese pizza', 'score': 0.9758967161178589}
{'product_name': 'Natural Cheese Pizza', 'cleaned_name': 'cheese pizza', 'score': 0.9758967161178589}
{'product_name': 'Cheese Pizza', 'cleaned_name': 'cheese pizza', 'score': 0.9758967161178589}
